# 01 Data Ingestion

## Purpose
Pull raw source data for the xwOBA pipeline and save it locally as parquet files.

## Inputs
- pybaseball Statcast API
- pybaseball sprint speed data
- local `wOBAbyYear.csv`

## Outputs
- `data/raw/statcast_events.parquet`
- `data/raw/sprint_speed.parquet`
- `data/raw/woba_weights.parquet`

In [1]:
import warnings
import pandas as pd
import os
import pybaseball as pb
from pybaseball.statcast_running import statcast_sprint_speed

In [2]:
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pb.cache.enable()

SEASONS = [2023, 2024, 2025]
DATE_RANGES = {
    2023: ("2023-03-29", "2023-10-02"),
    2024: ("2024-03-27", "2024-09-30"),
    2025: ("2025-03-26", "2025-09-29"),
}

## Pull Statcast data

In [3]:
dfs = []

for y in SEASONS:
    start_dt, end_dt = DATE_RANGES[y]
    print(f"Pulling {y} data...")
    df_y = pb.statcast(start_dt=start_dt, end_dt=end_dt)
    df_y["season"] = y
    dfs.append(df_y)
    
raw_statcast = pd.concat(dfs, ignore_index=True)

print("\nDone.")
print("Shape:", raw_statcast.shape)
raw_statcast.head()

Pulling 2023 data...
This is a large query, it may take a moment to complete


100%|██████████| 188/188 [01:01<00:00,  3.05it/s]


Pulling 2024 data...
This is a large query, it may take a moment to complete


100%|██████████| 188/188 [01:08<00:00,  2.74it/s]


Pulling 2025 data...
This is a large query, it may take a moment to complete


100%|██████████| 188/188 [00:52<00:00,  3.56it/s]



Done.
Shape: (2143789, 119)


,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,spin_dir,spin_rate_deprecated,break_angle_deprecated,break_length_deprecated,zone,des,game_type,stand,p_throws,home_team,away_team,type,hit_location,bb_type,balls,strikes,game_year,pfx_x,pfx_z,plate_x,plate_z,on_3b,on_2b,on_1b,outs_when_up,inning,inning_topbot,hc_x,hc_y,tfs_deprecated,tfs_zulu_deprecated,umpire,sv_id,vx0,vy0,vz0,ax,ay,az,sz_top,sz_bot,hit_distance_sc,launch_speed,launch_angle,effective_speed,release_spin_rate,release_extension,game_pk,fielder_2,fielder_3,fielder_4,fielder_5,fielder_6,fielder_7,fielder_8,fielder_9,release_pos_y,estimated_ba_using_speedangle,estimated_woba_using_speedangle,woba_value,woba_denom,babip_value,iso_value,launch_speed_angle,at_bat_number,pitch_number,pitch_name,home_score,away_score,bat_score,fld_score,post_away_score,post_home_score,post_bat_score,post_fld_score,if_fielding_alignment,of_fielding_alignment,spin_axis,delta_home_win_exp,delta_run_exp,bat_speed,swing_length,estimated_slg_using_speedangle,delta_pitcher_run_exp,hyper_speed,home_score_diff,bat_score_diff,home_win_exp,bat_win_exp,age_pit_legacy,age_bat_legacy,age_pit,age_bat,n_thruorder_pitcher,n_priorpa_thisgame_player_at_bat,pitcher_days_since_prev_game,batter_days_since_prev_game,pitcher_days_until_next_game,batter_days_until_next_game,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches,season
0,CH,2023-10-01,89.0,-2.8,5.59,"Robertson, Nick",677008,687798,field_out,hit_into_play,<NA>,<NA>,<NA>,<NA>,9,"Heston Kjerstad grounds out, first baseman Bob...",R,L,R,BAL,BOS,X,3,ground_ball,2,2,2023,-1.53,0.33,0.333018,2.005061,<NA>,<NA>,<NA>,2,9,Bot,158.28,166.83,<NA>,<NA>,<NA>,<NA>,11.122985,-129.176025,-3.49208,-19.471845,26.055263,-27.922064,3.81,1.74,6,96.4,-17,90.7,1703,7.4,716367,657136,666915,665839,622569,596115,677800,678882,608701,53.11,0.147,0.152,0.0,1,0,0,2,73,6,Changeup,1,6,1,6,6,1,1,6,Infield shade,Standard,250,-0.001,-0.233,66.5,6.5,0.149,0.233,96.4,-5,-5,0.001,0.001,24,24,25,24,1,3,11,1,<NA>,<NA>,2.55,1.53,-1.53,31.7,1.676715,-1.896554,41.830979,30.714944,26.41202,2023
1,FF,2023-10-01,96.9,-2.4,5.9,"Robertson, Nick",677008,687798,None,foul,<NA>,<NA>,<NA>,<NA>,5,Foul,R,L,R,BAL,BOS,S,<NA>,None,2,2,2023,-0.76,1.36,0.091181,2.705577,<NA>,<NA>,<NA>,2,9,Bot,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,8.558215,-140.741874,-6.148396,-12.116762,34.259201,-12.836434,3.81,1.74,223,78.2,56,98.4,2153,7.4,716367,657136,666915,665839,622569,596115,677800,678882,608701,53.13,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,73,5,4-Seam Fastball,1,6,1,6,6,1,1,6,Infield shade,Standard,211,0.0,0.0,70.8,6.7,<NA>,0.0,88.0,-5,-5,0.001,0.001,24,24,25,24,1,3,11,1,<NA>,<NA>,1.09,0.76,-0.76,47.4,8.715532,3.692542,40.551342,33.656454,26.020583,2023
2,CH,2023-10-01,90.0,-2.93,5.56,"Robertson, Nick",677008,687798,None,ball,<NA>,<NA>,<NA>,<NA>,13,Ball,R,L,R,BAL,BOS,B,<NA>,None,1,2,2023,-1.65,0.36,-0.24348,0.531787,<NA>,<NA>,<NA>,2,9,Bot,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,10.328033,-130.515462,-7.371558,-20.978432,27.339657,-26.599717,3.712182,1.775306,<NA>,<NA>,<NA>,91.5,1698,7.4,716367,657136,666915,665839,622569,596115,677800,678882,608701,53.14,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,73,4,Changeup,1,6,1,6,6,1,1,6,Infield shade,Standard,250,0.0,0.041,<NA>,<NA>,<NA>,-0.041,<NA>,-5,-5,0.001,0.001,24,24,25,24,1,3,11,1,<NA>,<NA>,2.47,1.65,-1.65,30.3,<NA>,<NA>,<NA>,<NA>,<NA>,2023
3,ST,2023-10-01,82.2,-3.09,5.55,"Robertson, Nick",677008,687798,None,ball,<NA>,<NA>,<NA>,<NA>,14,Ball,R,L,R,BAL,BOS,B,<NA>,None,0,2,2023,1.43,0.28,0.808145,0.486074,<NA>,<NA>,<NA>,2,9,Bot,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,6.108013,-119.483805,-5.435467,12.155221,26.646301,-28.491928,3.781975,1.739611,<NA>,<NA>,<NA>,82.4,2786,6.9,716367,657136,666915,665839,622569,596115,677800,678882,608701,53.63,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,73,3,Sweeper,1,6,1,6,6,1,1,6

In [4]:
raw_statcast.head(50)

,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,spin_dir,spin_rate_deprecated,break_angle_deprecated,break_length_deprecated,zone,des,game_type,stand,p_throws,home_team,away_team,type,hit_location,bb_type,balls,strikes,game_year,pfx_x,pfx_z,plate_x,plate_z,on_3b,on_2b,on_1b,outs_when_up,inning,inning_topbot,hc_x,hc_y,tfs_deprecated,tfs_zulu_deprecated,umpire,sv_id,vx0,vy0,vz0,ax,ay,az,sz_top,sz_bot,hit_distance_sc,launch_speed,launch_angle,effective_speed,release_spin_rate,release_extension,game_pk,fielder_2,fielder_3,fielder_4,fielder_5,fielder_6,fielder_7,fielder_8,fielder_9,release_pos_y,estimated_ba_using_speedangle,estimated_woba_using_speedangle,woba_value,woba_denom,babip_value,iso_value,launch_speed_angle,at_bat_number,pitch_number,pitch_name,home_score,away_score,bat_score,fld_score,post_away_score,post_home_score,post_bat_score,post_fld_score,if_fielding_alignment,of_fielding_alignment,spin_axis,delta_home_win_exp,delta_run_exp,bat_speed,swing_length,estimated_slg_using_speedangle,delta_pitcher_run_exp,hyper_speed,home_score_diff,bat_score_diff,home_win_exp,bat_win_exp,age_pit_legacy,age_bat_legacy,age_pit,age_bat,n_thruorder_pitcher,n_priorpa_thisgame_player_at_bat,pitcher_days_since_prev_game,batter_days_since_prev_game,pitcher_days_until_next_game,batter_days_until_next_game,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches,season
0,CH,2023-10-01,89.0,-2.8,5.59,"Robertson, Nick",677008,687798,field_out,hit_into_play,<NA>,<NA>,<NA>,<NA>,9,"Heston Kjerstad grounds out, first baseman Bob...",R,L,R,BAL,BOS,X,3,ground_ball,2,2,2023,-1.53,0.33,0.333018,2.005061,<NA>,<NA>,<NA>,2,9,Bot,158.28,166.83,<NA>,<NA>,<NA>,<NA>,11.122985,-129.176025,-3.49208,-19.471845,26.055263,-27.922064,3.81,1.74,6,96.4,-17,90.7,1703,7.4,716367,657136,666915,665839,622569,596115,677800,678882,608701,53.11,0.147,0.152,0.0,1,0,0,2,73,6,Changeup,1,6,1,6,6,1,1,6,Infield shade,Standard,250,-0.001,-0.233,66.5,6.5,0.149,0.233,96.4,-5,-5,0.001,0.001,24,24,25,24,1,3,11,1,<NA>,<NA>,2.55,1.53,-1.53,31.7,1.676715,-1.896554,41.830979,30.714944,26.41202,2023
1,FF,2023-10-01,96.9,-2.4,5.9,"Robertson, Nick",677008,687798,None,foul,<NA>,<NA>,<NA>,<NA>,5,Foul,R,L,R,BAL,BOS,S,<NA>,None,2,2,2023,-0.76,1.36,0.091181,2.705577,<NA>,<NA>,<NA>,2,9,Bot,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,8.558215,-140.741874,-6.148396,-12.116762,34.259201,-12.836434,3.81,1.74,223,78.2,56,98.4,2153,7.4,716367,657136,666915,665839,622569,596115,677800,678882,608701,53.13,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,73,5,4-Seam Fastball,1,6,1,6,6,1,1,6,Infield shade,Standard,211,0.0,0.0,70.8,6.7,<NA>,0.0,88.0,-5,-5,0.001,0.001,24,24,25,24,1,3,11,1,<NA>,<NA>,1.09,0.76,-0.76,47.4,8.715532,3.692542,40.551342,33.656454,26.020583,2023
2,CH,2023-10-01,90.0,-2.93,5.56,"Robertson, Nick",677008,687798,None,ball,<NA>,<NA>,<NA>,<NA>,13,Ball,R,L,R,BAL,BOS,B,<NA>,None,1,2,2023,-1.65,0.36,-0.24348,0.531787,<NA>,<NA>,<NA>,2,9,Bot,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,10.328033,-130.515462,-7.371558,-20.978432,27.339657,-26.599717,3.712182,1.775306,<NA>,<NA>,<NA>,91.5,1698,7.4,716367,657136,666915,665839,622569,596115,677800,678882,608701,53.14,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,73,4,Changeup,1,6,1,6,6,1,1,6,Infield shade,Standard,250,0.0,0.041,<NA>,<NA>,<NA>,-0.041,<NA>,-5,-5,0.001,0.001,24,24,25,24,1,3,11,1,<NA>,<NA>,2.47,1.65,-1.65,30.3,<NA>,<NA>,<NA>,<NA>,<NA>,2023
3,ST,2023-10-01,82.2,-3.09,5.55,"Robertson, Nick",677008,687798,None,ball,<NA>,<NA>,<NA>,<NA>,14,Ball,R,L,R,BAL,BOS,B,<NA>,None,0,2,2023,1.43,0.28,0.808145,0.486074,<NA>,<NA>,<NA>,2,9,Bot,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,6.108013,-119.483805,-5.435467,12.155221,26.646301,-28.491928,3.781975,1.739611,<NA>,<NA>,<NA>,82.4,2786,6.9,716367,657136,666915,665839,622569,596115,677800,678882,608701,53.63,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,73,3,Sweeper,1,6,1,6,6,1,1,6

## Pull sprint speed data

In [5]:
ss = pd.concat(
    [statcast_sprint_speed(y, min_opp=10).assign(season=y) for y in SEASONS],
    ignore_index=True
)

print("Sprint speed shape:", ss.shape)
ss.head()

Sprint speed shape: (1728, 11)


,"last_name, first_name",player_id,team_id,team,position,age,competitive_runs,bolts,hp_to_1b,sprint_speed,season
0,"De La Cruz, Elly",682829,113,CIN,SS,21,162,84.0,4.13,30.5,2023
1,"Witt Jr., Bobby",677951,118,KC,SS,23,262,149.0,4.12,30.5,2023
2,"Blanco, Dairon",680118,118,KC,RF,30,55,31.0,4.15,30.3,2023
3,"Turner, Trea",607208,143,PHI,SS,30,288,115.0,4.14,30.3,2023
4,"Thompson, Bubba",669352,140,TEX,LF,25,28,19.0,4.23,30.2,2023


## Load wOBA weights

In [6]:
woba = pd.read_csv("wOBAbyYear.csv")
woba = woba.rename(columns={
    "Season": "season",
    "wBB": "unintentional_bb",
    "wHBP": "hbp",
    "w1B": "single_weight",
    "w2B": "double_weight",
    "w3B": "triple_weight",
    "wHR": "hr_weight"
})
woba = woba[[
    "season",
    "unintentional_bb",
    "hbp",
    "single_weight",
    "double_weight",
    "triple_weight",
    "hr_weight"
]].copy()
woba["season"] = pd.to_numeric(woba["season"], errors="coerce").astype("Int64")
for col in [
    "unintentional_bb",
    "hbp",
    "single_weight",
    "double_weight",
    "triple_weight",
    "hr_weight"
]: woba[col] = pd.to_numeric(woba[col], errors="coerce")

woba = woba[woba["season"].isin(SEASONS)].copy()
woba = woba.sort_values("season").reset_index(drop=True)

print("wOBA weights shape:", woba.shape)
display(woba)

wOBA weights shape: (3, 7)


,season,unintentional_bb,hbp,single_weight,double_weight,triple_weight,hr_weight
0,2023,0.696,0.726,0.883,1.244,1.569,2.004
1,2024,0.689,0.720,0.882,1.254,1.590,2.050
2,2025,0.691,0.722,0.882,1.252,1.584,2.037


## Save raw datasets locally

In [7]:
os.makedirs("data/raw", exist_ok=True)

raw_statcast.to_parquet("data/raw/statcast_events.parquet", index=False)
ss.to_parquet("data/raw/sprint_speed.parquet", index=False)
woba.to_parquet("data/raw/woba_weights.parquet", index=False)

print("All raw datasets saved.")

All raw datasets saved.
